# Explorare: rețeaua, perechile OD și vulnerabilitatea (P3)

Notebook pentru etapele M1 și M2. Dacă graful din pachet nu e încă în `data/`, se folosește o grilă sintetică.

In [ ]:
import os, sys
from pathlib import Path
# notebook-ul se poate deschide din notebooks/ sau din rădăcină; lucrăm mereu din rădăcină
if not Path("config/default.yaml").exists() and Path("../config/default.yaml").exists():
    os.chdir("..")
sys.path.insert(0, "src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sada.config import load_config
cfg = load_config("config/default.yaml")
print("Lucrez în", Path.cwd())

In [ ]:
import networkx as nx
from sada.p3.graph_io import load_graph, summarize
gpath = Path(cfg["data"]["graph"])
if gpath.exists():
    G = load_graph(gpath)
else:
    print("Nu găsesc graful din pachet; folosesc o grilă SINTETICĂ 20×20.")
    G = nx.MultiDiGraph(nx.grid_2d_graph(20, 20).to_directed())
    G = nx.relabel_nodes(G, {n: f"{n[0]}_{n[1]}" for n in G.nodes})
    rng = np.random.default_rng(0)
    for n in G.nodes:
        i, j = map(int, n.split("_")); G.nodes[n].update(x=float(i), y=float(j), population=float(rng.integers(0, 100)))
    for u, v, k in G.edges(keys=True): G[u][v][k]["length"] = float(rng.uniform(80, 200))
summarize(G)

## 1. Structura: grade și lungimi

O rețea rutieră are puține noduri cu grad mare (intersecții importante) și multe noduri de grad 2–4. Muchiile foarte lungi sunt adesea poduri sau artere, adică puncte vulnerabile.

In [ ]:
deg = pd.Series(dict(G.degree()))
lengths = pd.Series([d.get("length", np.nan) for _, _, d in G.edges(data=True)])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2))
deg.value_counts().sort_index().plot.bar(ax=a1); a1.set_xlabel("Grad (intrări + ieșiri)"); a1.set_ylabel("Noduri")
lengths.plot.hist(bins=40, ax=a2); a2.set_xlabel("Lungimea muchiei [m]")
plt.tight_layout(); plt.show()
print(f"lungime medie {lengths.mean():.0f} m, cele mai lungi 1%: peste {lengths.quantile(0.99):.0f} m")

## 2. Primul experiment de vulnerabilitate, fără recuperare

Ce procent din perechi rămân conectate când scoatem progresiv muchii, aleator vs. după importanță. Varianta de aici e minimală, pentru intuiție; implementarea voastră din `failures.py` o înlocuiește în experimente.

In [ ]:
from sada.p3.od import sample_od_pairs
from sada.p3.metrics import od_costs
from sada.p3.failures import edge_importance
pairs = sample_od_pairs(G, 150, np.random.default_rng(1))
imp = edge_importance(G, k=min(200, G.number_of_nodes()))
edges = list(G.edges(keys=True))
rng = np.random.default_rng(2)
rand_order = [edges[i] for i in rng.permutation(len(edges))]
targ_order = sorted(imp, key=imp.get, reverse=True)
fracs = [0, 0.02, 0.04, 0.06, 0.08, 0.10]
res = {"aleator": [], "țintit": []}
for name, order in (("aleator", rand_order), ("țintit", targ_order)):
    for f in fracs:
        H = nx.restricted_view(G, [], order[: int(f * len(edges))])
        res[name].append(100 * np.isfinite(od_costs(H, pairs)).mean())
pd.DataFrame(res, index=[f"{100*f:.0f}%" for f in fracs]).plot(marker="o")
plt.xlabel("Muchii eliminate"); plt.ylabel("Perechi OD conectate [%]"); plt.title("Fără recuperare"); plt.show()

## 3. Câte perechi au o rută de rezervă?

Merge după ce implementați `routes.backup_route` (GHID.md, pasul 3).

In [ ]:
from sada.p3.routes import backup_route, primary_route
from sada.p3.graph_io import node_key
try:
    no_backup = 0
    for _, r in pairs.head(60).iterrows():
        o, d = node_key(G, r.origin), node_key(G, r.destination)
        if backup_route(G, o, d, primary_route(G, o, d)) is None:
            no_backup += 1
    print(f"{no_backup} din 60 de perechi nu au rută de rezervă fără muchii comune")
except NotImplementedError as e:
    print("Încă nu e implementat:", e)

## Ce notați pentru nota de proiectare

- dimensiunea grafului și dacă e orientat și tare conex;
- cât de mult diferă curbele aleator / țintit (motivația nivelului 2);
- câte perechi nu au rezervă (motivația nivelului 3).